# Let's play around with period detection

We will use:
- Discrete Cosine Transforms


**NOTE:** This will be done on mainly the raw data!

In [6]:
from circadian_cell_painting import(
    detect,
    OscillationResults,
    cosinor_fitted_long
)
from circadian_cell_painting.osc_detection import DampedCosinorDetection

from pathlib import Path
import polars as pl
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
import string
from tqdm import tqdm
import numpy as np
from scipy.signal import savgol_filter
import pandas as pd
from natsort import natsorted
import matplotlib.colors as mcolors
from scipy.fftpack import dct
import numpy.typing as npt

raw_data = Path("/home/derfelt/PartchLabFiles/Circadian "
                "Rhythms/Live Study/enVision/enVision_fo"
                "rmatted/2026-05-11._.FORMATTED.tsv")


raw_df = pl.read_csv(raw_data, separator="\t")

platemap = (
    pl.read_excel("/home/derfelt/PartchLabFiles/Circadian "
                  "Rhythms/Live Study/platemaps/2026-05-11"
                  "Echo_transfer.xlsx", engine='openpyxl')
    .select(["384_Well", 
             "MoleculeID", 
             "strain",
             "sample_type"])
    .unique(subset=["384_Well"])
    .with_columns(
        pl.col("MoleculeID").str.strip_chars()
    )
).rename({"384_Well": "Well"})

meta_cols = ["MoleculeID", "strain"]
wells = raw_df['Well'].unique().to_list()

raw_df = raw_df.join(platemap, on='Well', how='left')
raw_df = (
    raw_df.with_columns(
        pl.concat_str(
            [pl.col(c).fill_null('nan').cast(pl.String) for c in meta_cols],
            separator=' | ',
        ).alias('group_label')
    )
)


display(raw_df)

Well,CalcResultI,Signal,Timepoint,MoleculeID,strain,sample_type,group_label
str,i64,i64,f64,str,str,str,str
"""A01""",20,4,0.0,null,null,null,"""nan | nan"""
"""A02""",20,4,0.0,null,null,null,"""nan | nan"""
"""A03""",20,4,0.0,null,null,null,"""nan | nan"""
"""A04""",40,8,0.0,null,null,null,"""nan | nan"""
"""A05""",40,8,0.0,null,null,null,"""nan | nan"""
…,…,…,…,…,…,…,…
"""P20""",20,4,164.0,null,null,null,"""nan | nan"""
"""P21""",20,4,164.0,null,null,null,"""nan | nan"""
"""P22""",20,4,164.0,null,null,null,"""nan | nan"""


In [5]:
display(raw_df.filter(pl.col('group_label').str.contains('3TP', literal=True)))

Well,CalcResultI,Signal,Timepoint,MoleculeID,strain,sample_type,group_label
str,i64,i64,f64,str,str,str,str
"""H13""",3420,684,0.0,"""Dex_Chroma_3TP_bmal1""","""bmal1""",null,"""Dex_Chroma_3TP_bmal1 | bmal1"""
"""H14""",3760,752,0.0,"""Dex_Chroma_3TP_bmal1""","""bmal1""",null,"""Dex_Chroma_3TP_bmal1 | bmal1"""
"""H15""",3820,764,0.0,"""Dex_Chroma_3TP_bmal1""","""bmal1""",null,"""Dex_Chroma_3TP_bmal1 | bmal1"""
"""I13""",10660,2132,0.0,"""Dex_PhenoVue_3TP_per2""","""per2""",null,"""Dex_PhenoVue_3TP_per2 | per2"""
"""I14""",10100,2020,0.0,"""Dex_PhenoVue_3TP_per2""","""per2""",null,"""Dex_PhenoVue_3TP_per2 | per2"""
…,…,…,…,…,…,…,…
"""H14""",820,164,164.0,"""Dex_Chroma_3TP_bmal1""","""bmal1""",null,"""Dex_Chroma_3TP_bmal1 | bmal1"""
"""H15""",820,164,164.0,"""Dex_Chroma_3TP_bmal1""","""bmal1""",null,"""Dex_Chroma_3TP_bmal1 | bmal1"""
"""I13""",4600,920,164.0,"""Dex_PhenoVue_3TP_per2""","""per2""",null,"""Dex_PhenoVue_3TP_per2 | per2"""


In [7]:
def dct_period_detection(values: npt.NDArray[np.float64], 
                         sampling_interval: float) -> tuple[npt.NDArray, npt.NDArray, npt.NDArray]:
    """
    Runs a discrete cosinor 

    Args:
        values (npt.NDArray[np.float64]): Timeseries measurement (not timepoints)
        sampling_interval (float): the amount of reads per hour

    Returns:
        tuple[npt.NDArray, npt.NDArray, npt.NDArray]: returns the periods, frequency, and amplitudes
    """
    transformed = dct(values, norm="ortho")
    
    # grab the amplitude (the abs of the transformed data)
    amplitude = np.abs(transformed)
    
    N = len(values)
    k = np.arange(N)
    
    # sampling frequency
    fs = 1 / sampling_interval
    
    f = fs * k / (2 * N)
    
    # convert to periods
    
    with np.errstate(divide='ignore'):
        periods = np.where(f > 0, 1 / f, np.inf)
    
    # print(periods, f, amplitude)
    
    return periods, f, amplitude
    

In [10]:
def add_period_marks(ax, x_min, x_max, interval=12):
    marks = np.arange(np.ceil(x_min / interval) * interval, x_max + interval, interval)
    for mark in marks:
        ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
    ax.set_xticks(marks)
    ax.set_xticklabels([f'{int(m)}h' for m in marks], fontsize=7)

well_col = "Well"
timepoint_col = "Timepoint"
feature_col = 'CalcResultI'
sample_interval = 0.5

period_range = (22, 48)

wells = raw_df['Well'].unique().to_list()
group_labels = raw_df['group_label'].unique().to_list()
# print(*group_labels, sep='\n')

out_path = Path("/home/derfelt/PartchLabFiles/Circadian "
                "Rhythms/Live Study/enVision/enVision_vis"
                "ual_results/2026-05-11")
out_path = out_path / "dct_plots.pdf"

best_periods = []
with PdfPages(out_path) as pdf:
    for group in tqdm(natsorted(group_labels), desc="replicate groups"):
        group_df = (raw_df
                    .filter(pl.col('group_label') == group)
                    )
        group_wells = group_df[well_col].unique().to_list()
        
        # DCT replicates
        detected_periods = []
        all_amplitudes = []
        all_periods_mask = None
        for well in group_wells:
            well_data = (group_df
                         .filter(pl.col(well_col) == well)
                         .sort(timepoint_col)[feature_col]
                         .to_numpy()
                         )

            period, freq, amplitude = dct_period_detection(well_data, sample_interval)
            mask = (period >= period_range[0]) & (period <= period_range[1])
            
            all_amplitudes.append(amplitude[mask])
            all_periods_mask = period[mask]
            
            # detected period
            peak_idx = np.argmax(amplitude[mask])
            detected_period = period[mask][peak_idx]
            detected_periods.append(detected_period)
            
        all_amplitudes = np.array(all_amplitudes)
        mean_amplitudes = np.mean(all_amplitudes, axis=0)
        std_amplitudes = np.std(all_amplitudes, axis=0)
            
        mean_period = np.mean(detected_periods)
        std_period = np.std(detected_periods)
        median_period = np.median(detected_periods)
        
        best_periods.append({
            'group_label': group,
            'mean_period': mean_period,
            'median_period': median_period,
            'std_period': std_period,
            'n_replicates': len(group_wells),
            'periods': detected_periods
        })
        
        fig, ax = plt.subplots(figsize=(14,5))
        
        
        ax.plot(all_periods_mask, mean_amplitudes, color='blue')
        ax.fill_between(
            all_periods_mask,
            mean_amplitudes - std_amplitudes,
            mean_amplitudes + std_amplitudes,
            alpha=0.2,
            color='blue'
        )
        
        ax.axvline(mean_period, color='red', linewidth=2,
                   label=f'Mean period: {mean_period:.1f} ± {std_period:.1f}h')
        ax.axvline(median_period, color='orange', linewidth=2, linestyle='--',
                   label=f'median period: {median_period:.1f}h', alpha=0.5)
        ax.axvspan(mean_period - std_period, mean_period + std_period,
                   alpha=0.1, color='red')  # ± 1 std shading
        
        ax.set_xlim(period_range[0], period_range[1])
        
        ax.set_xlabel('Period (hours)')
        ax.set_ylabel('Amplitude')
        ax.set_title(f'{group}\nDetected period: {mean_period:.1f} ± {std_period:.1f}h\n'
                     f'detected median: {median_period:.1f}h')
        ax.legend(fontsize=7)
        add_period_marks(ax, period_range[0], period_range[1], interval=1)
        
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

best_periods = pl.DataFrame(best_periods)
display(best_periods)

csv_out = Path("/home/derfelt/PartchLabFiles/Circadian "
               "Rhythms/Live Study/enVision/period_dete"
               "ction_results")
csv_out = csv_out / "5-11_period_results._.DEX.csv"

(best_periods
 .filter(pl.col("group_label").str.contains("TP", literal=True))
 .drop("periods")
 .write_csv(csv_out))


csv_out = Path("/home/derfelt/PartchLabFiles/Circadian "
               "Rhythms/Live Study/enVision/period_dete"
               "ction_results")
csv_out = csv_out / "5-11_period_results.csv"

(best_periods
 .drop("periods")
 .write_csv(csv_out))


replicate groups: 100%|██████████| 48/48 [00:05<00:00,  9.24it/s]


group_label,mean_period,median_period,std_period,n_replicates,periods
str,f64,f64,f64,i64,list[f64]
"""0.25% DMSO | bmal1""",29.727273,29.727273,0.0,3,"[29.727273, 29.727273, 29.727273]"
"""0.25% DMSO | per2""",25.153846,25.153846,3.5527e-15,3,"[25.153846, 25.153846, 25.153846]"
"""0.0078% DMSO | bmal1""",29.727273,29.727273,0.0,3,"[29.727273, 29.727273, 29.727273]"
"""0.0078% DMSO | per2""",31.741758,25.153846,10.612555,3,"[25.153846, 46.714286, 23.357143]"
"""0.125% DMSO | bmal1""",29.727273,29.727273,0.0,3,"[29.727273, 29.727273, 29.727273]"
…,…,…,…,…,…
"""MK-8800 412.5nM | per2""",32.340659,25.153846,10.163689,3,"[25.153846, 46.714286, 25.153846]"
"""MK-8800 825nM | bmal1""",29.727273,29.727273,0.0,3,"[29.727273, 29.727273, 29.727273]"
"""MK-8800 825nM | per2""",32.340659,25.153846,10.163689,3,"[25.153846, 46.714286, 25.153846]"


# Lets figure out the consensus window for all the entire experiment

Devons suggested that we find the max period and use that as the window for all
the samples

In [12]:
cols = ['mean_period', 'median_period']

consensus_df = (best_periods
                .select(cols)
                .unpivot(variable_name='metrics', value_name='values')
                .group_by('metrics')
                .agg([
                    pl.col("values").mean().alias("mean"),
                    pl.col("values").median().alias("median")
                ])
                )

display(consensus_df)

metrics,mean,median
str,f64,f64
"""mean_period""",29.139433,29.727273
"""median_period""",28.463767,29.727273


Based on the results it looks like the window is **30** hours

# Lets combine results with raw data + detrend

**NOTE**: The best way to do this is probably by detecting the period for each well and then have the option to detrend based on the replicate groups periods

In [ ]:
raw_data_periods =  raw_df.join(best_periods, on='group_label', how='left').sort([well_col, timepoint_col])

display(raw_data_periods)

Well,CalcResultI,Signal,Timepoint,MoleculeID,strain,sample_type,group_label,mean_period,median_period,std_period,n_replicates,periods
str,i64,i64,f64,str,str,str,str,f64,f64,f64,i64,list[f64]
"""A01""",20,4,0.0,null,null,null,"""nan | nan""",32.064656,31.0,7.487103,244,"[27.9, 27.9, … 46.5]"
"""A01""",20,4,0.5,null,null,null,"""nan | nan""",32.064656,31.0,7.487103,244,"[27.9, 27.9, … 46.5]"
"""A01""",20,4,1.0,null,null,null,"""nan | nan""",32.064656,31.0,7.487103,244,"[27.9, 27.9, … 46.5]"
"""A01""",20,4,1.5,null,null,null,"""nan | nan""",32.064656,31.0,7.487103,244,"[27.9, 27.9, … 46.5]"
"""A01""",20,4,2.0,null,null,null,"""nan | nan""",32.064656,31.0,7.487103,244,"[27.9, 27.9, … 46.5]"
…,…,…,…,…,…,…,…,…,…,…,…,…
"""P24""",40,8,136.5,"""food coloring""",null,null,"""food coloring | nan""",31.553571,31.553571,8.303571,2,"[23.25, 39.857143]"
"""P24""",20,4,137.0,"""food coloring""",null,null,"""food coloring | nan""",31.553571,31.553571,8.303571,2,"[23.25, 39.857143]"
"""P24""",40,8,137.5,"""food coloring""",null,null,"""food coloring | nan""",31.553571,31.553571,8.303571,2,"[23.25, 39.857143]"


Detrending data

In [41]:
well_col = "Well"
timepoint_col = "Timepoint"
feature_col = 'CalcResultI'
group_col = 'group_label'
sample_interval = 0.5

detrended_data = []
groups = raw_data_periods[group_col].unique().to_list()
for group in tqdm(sorted(groups), desc='exp groupings', disable=True):
    group_df = (raw_data_periods
                .filter(pl.col(group_col) == group)
    )
    
    median_period = group_df["median_period"][0]
    window_size = round(median_period / sample_interval)
    if window_size % 2 == 0:
        window_size += 1
    
    print(f"{group}: median_period = {median_period:.1f}h | window_size = {window_size} tpoints")
    
    pass1 = [pl.col(feature_col)
             .rolling_mean(window_size=window_size, center=True, min_samples=window_size)
             .over(well_col)
             .alias(f'{feature_col}_pass1')
             ]
    
    pass2 = [pl.col(f'{feature_col}_pass1')
             .rolling_mean(window_size=window_size, center=True, min_samples=window_size)
             .over(well_col)
             .alias(f'{feature_col}_pass2')
             ]
    
    detrend = [(pl.col(feature_col) - pl.col(f'{feature_col}_pass2'))]
    
    temp_cols = [f"{feature_col}_pass1", f"{feature_col}_pass2"]
    
    detrended_group = (
        group_df
        .sort([well_col, timepoint_col])
        .with_columns(pass1)
        .with_columns(pass2)
        .with_columns(detrend)
        .drop(temp_cols)
        .drop_nulls(subset=[feature_col])
    )
    
    detrended_data.append(detrended_group)
    
    print(f"min={detrended_group[timepoint_col].min()}, max={detrended_group[timepoint_col].max()}h")


detrended_data = pl.concat(detrended_data).sort([well_col, timepoint_col])

# display(detrended_data)

0.0078% DMSO | bmal1: median_period = 31.0h | window_size = 63 tpoints
min=31.0, max=107.5h
0.0078% DMSO | per2: median_period = 25.4h | window_size = 51 tpoints
min=25.0, max=113.5h
0.0156% DMSO | bmal1: median_period = 31.0h | window_size = 63 tpoints
min=31.0, max=107.5h
0.0156% DMSO | per2: median_period = 25.4h | window_size = 51 tpoints
min=25.0, max=113.5h
0.03125% DMSO | bmal1: median_period = 27.9h | window_size = 57 tpoints
min=28.0, max=110.5h
0.03125% DMSO | per2: median_period = 25.4h | window_size = 51 tpoints
min=25.0, max=113.5h
0.0625% DMSO | bmal1: median_period = 31.0h | window_size = 63 tpoints
min=31.0, max=107.5h
0.0625% DMSO | per2: median_period = 25.4h | window_size = 51 tpoints
min=25.0, max=113.5h
0.125% DMSO | bmal1: median_period = 31.0h | window_size = 63 tpoints
min=31.0, max=107.5h
0.125% DMSO | per2: median_period = 25.4h | window_size = 51 tpoints
min=25.0, max=113.5h
0.25% DMSO | bmal1: median_period = 31.0h | window_size = 63 tpoints
min=31.0, max=10

In [42]:
def make_title(row: dict, well_col: str, meta_cols: list[str]) -> str:
    parts = [row[well_col]]
    for col in meta_cols:
        val = row.get(col)
        if val is not None and str(val).strip() != '' and str(val).lower() != 'nan':
            parts.append(str(val))
    return ' | '.join(parts)

In [44]:
# visualizing detrended data

meta_cols = ["MoleculeID", "strain"]

# smoothing option
smoothed_detrended_data = (detrended_data
           .group_by(well_col)
           .map_groups(
               lambda g: 
                   g.with_columns(
                       pl.Series(feature_col, savgol_filter(
                           g[feature_col].to_numpy(),
                           window_length=13,
                           polyorder=3
                       ))
                       )
                   )
           .sort(["Well", "Timepoint"])
)

display(smoothed_detrended_data)

wells = smoothed_detrended_data["Well"].unique().sort().to_list()
# out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/enVision_visual_results/detrended_savgol_smoothed_timeseries.pdf")

out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/enVision/enVision_visual_results/2026-05-11")
out_path = out_path / "DCT-INFORMED._.detrended_savgol_smoothed_timeseries.pdf"

# timepoint_col = "Timepoint"
# value = "mean"
# meta_cols = ['MoleculeID', 'strain', 'dex']

with PdfPages(out_path) as pdf:
    g = sns.FacetGrid(
        data=smoothed_detrended_data,
        col="Well",
        col_wrap=24,
        height=2.5,
        aspect=1.8,
        sharey=False,
        sharex=False,
    )
    g.map_dataframe(sns.lineplot, x=timepoint_col, y=feature_col, color="red", linewidth=1.5)
    
    # add marks and set ticks after mapping
    x_min = smoothed_detrended_data['Timepoint'].min()
    x_max = smoothed_detrended_data['Timepoint'].max()
    tick_marks = np.arange(np.ceil(x_min / 12) * 12, x_max + 12, 12)

    for ax in g.axes.flat:
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_xticks(tick_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in tick_marks], fontsize=6)
        for mark in tick_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
        
        
    g.set_titles(col_template='Well {col_name}')
    g.set_axis_labels('Time (hours)', 'Detrended signal')
    g.tight_layout()
    pdf.savefig(g.figure)
    plt.close()
    
    
    for well in tqdm(wells, total=len(wells), desc="Well progress"):
        well_df = (smoothed_detrended_data
              .filter((pl.col("Well") == well))
              .sort("Timepoint", descending=False)
              .to_pandas()
              )
        
        fig, ax = plt.subplots(figsize=(12, 4))
        
        # get first row for metadata
        row = well_df.iloc[0].to_dict()
        title = make_title(row, well_col='Well', meta_cols=meta_cols)
        
        median_period = well_df["median_period"][0]
        
        # sns.lineplot(data=well_df, x=timepoint_col, y=value, ax=ax, color="red", linewidth=2 marker="o", markersize=2)
        sns.lineplot(data=well_df, x=timepoint_col, y=feature_col, ax=ax, color="red", linewidth=2)
        
        
        ax.set_title(f"{title}\n"
                     f"median period: {median_period:.1f}h")
        ax.set_xlabel("Timepoint")
        ax.set_ylabel("Expression")
        
        x_min = well_df[timepoint_col].min()
        x_max = well_df[timepoint_col].max()
        period_marks = np.arange(np.ceil(x_min/12) * 12, x_max + 12, 12)
        
        for mark in period_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
        ax.set_xticks(period_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in period_marks])
        ax.axhline(0, linestyle='--', color='gray', alpha=0.2)
        
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        

Well,CalcResultI,Signal,Timepoint,MoleculeID,strain,sample_type,group_label,mean_period,median_period,std_period,n_replicates,periods
str,f64,i64,f64,str,str,str,str,f64,f64,f64,i64,list[f64]
"""A01""",-19.311353,0,31.0,null,null,null,"""nan | nan""",32.064656,31.0,7.487103,244,"[27.9, 27.9, … 46.5]"
"""A01""",-11.94477,4,31.5,null,null,null,"""nan | nan""",32.064656,31.0,7.487103,244,"[27.9, 27.9, … 46.5]"
"""A01""",-6.939246,4,32.0,null,null,null,"""nan | nan""",32.064656,31.0,7.487103,244,"[27.9, 27.9, … 46.5]"
"""A01""",-3.911171,4,32.5,null,null,null,"""nan | nan""",32.064656,31.0,7.487103,244,"[27.9, 27.9, … 46.5]"
"""A01""",-2.476932,4,33.0,null,null,null,"""nan | nan""",32.064656,31.0,7.487103,244,"[27.9, 27.9, … 46.5]"
…,…,…,…,…,…,…,…,…,…,…,…,…
"""P24""",-4.205358,4,105.5,"""food coloring""",null,null,"""food coloring | nan""",31.553571,31.553571,8.303571,2,"[23.25, 39.857143]"
"""P24""",-1.186629,4,106.0,"""food coloring""",null,null,"""food coloring | nan""",31.553571,31.553571,8.303571,2,"[23.25, 39.857143]"
"""P24""",1.990896,4,106.5,"""food coloring""",null,null,"""food coloring | nan""",31.553571,31.553571,8.303571,2,"[23.25, 39.857143]"


Well progress: 100%|██████████| 384/384 [00:47<00:00,  8.09it/s]
